<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/labs/Module%204/logo.png" width="300" alt="cognitiveclass.ai logo">
</center>


# **Building AI Agents from Scratch with Python**


Estimated time needed: **30** minutes


Build AI agents from scratch in just 30 minutes and use them to power real-world applications like a **Weather AI Agent** and **The Daily Dish customer-service chatbot**. In this hands-on project, you’ll apply in-demand AI skills by leveraging LLMs to solve practical challenges, create and orchestrate multi-agent systems, integrate memory and databases, and refine prompt engineering. This lab is essential for software engineers and machine learning engineers looking to level up their AI agent development skills.


## Lab Objectives

By the end of this lab, you'll be able to:

- Understand the core concepts of **AI Agents** and **Multi-Agent Systems (MAS)**.

- Build a **Weather AI Agent from scratch using Python** to retrieve and process external data.

- Develop a customer-service chatbot for **The Daily Dish** that answers reservation and menu-related questions using a PDF-based knowledge source.

- Implement agent memory to retain context and enable more personalized interactions.

- Design and integrate a **Multi-agent architecture** where agents collaborate to solve real-world problems.


## Agent Type 1: Weather AI Agent

Here’s your challenge: You’ve been hired to build a smart weather assistant for a company that needs quick and accurate weather updates. Checking weather information manually takes time and can lead to mistakes. Your task is to create a Weather AI Agent that can fetch current weather data, remember previous requests, and provide clear responses to users.

You’ll solve this by building specialized AI agents that work together—one to retrieve weather data, another to manage memory—so the system can respond efficiently and intelligently.


### Project Overview: Weather AI Agent Workflow 

In this project, you will build a Weather AI Agent using LLM-based agents that work together to provide weather information. The system uses a multi-agent design with memory to deliver smarter, more contextual responses.

1. **Weather Retrieval Phase (Agent 1):**
The user’s weather-related question is sent to a dedicated weather retrieval agent. This agent’s responsibility is to fetch real-time weather data (such as temperature and conditions) from an external weather source.

2. **Memory Phase (Agent 2):**
The retrieved weather information is passed to a memory agent. This agent’s role is to store and recall past weather queries and responses, enabling the system to maintain context across interactions.

3. **Response Generation Phase (Agent 3 – LLM):**
The weather data, along with any relevant memory context, is sent to an LLM-based response agent. This agent’s role is to generate a clear, natural, and user-friendly weather response.

#### Weather Agent with Memory — Diagram-Style Workflow (LLM-Based, Multi-Agent System)
```
┌──────────────┐
│   User       │
│ (Weather Q)  │
└──────┬───────┘
       │
       ▼
┌───────────────────────┐
│ Weather Retrieval     │
│ Agent (Agent 1)       │
│ - Calls Weather API   │
│ - Fetches live data   │
└──────┬────────────────┘
       │
       ▼
┌───────────────────────┐
│ Memory Agent          │
│ (Agent 2)             │
│ - Stores past queries │
│ - Recalls context     │
└──────┬────────────────┘
       │
       ▼
┌────────────────────────────┐
│ LLM Response Agent         │
│ (Agent 3)                  │
│ - Combines weather data    │
│ - Uses memory context      │
│ - Generates natural reply  │
└──────┬─────────────────────┘
       │
       ▼
┌──────────────┐
│   User       │
│ (Final Reply)│
└──────────────┘
```


---


### **Setup and prerequisites**

#### Step 0 — Get Your OpenWeather API Key:
Before writing any code, 
1. Open OpenWeather `https://home.openweathermap.org/` and **Register/Log in** to your account.
2. Navigate to API Keys `https://home.openweathermap.org/api_keys` and copy your **API key**, as shown in the screenshot below.

<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/KlRR3k8c739PEjzS4nTUHw/openweather-api.png" >
</center>


In [ ]:
# Copy-Paste the API code here
WEATHER_API_KEY = "86de44e6f84d828f3093dd2e1f6845dd"

#### Install required packages:


In [ ]:
!pip install requests python-dotenv

<span style="color:red"><b>Restart the kernel once the installation is complete.</b></span>


#### Weather Retrieval Agent


In [ ]:
import requests

class WeatherRetrievalAgent:
    def __init__(self, api_key):
        self.api_key = api_key
        self.base_url = "http://api.openweathermap.org/data/2.5/weather"

    def get_weather(self, city):
        params = {"q": city, "appid": self.api_key, "units": "metric"}
        res = requests.get(self.base_url, params=params)
        if res.status_code == 200:
            data = res.json()
            return {
                "city": city,
                "temperature": data["main"]["temp"],
                "humidity": data["main"]["humidity"],
                "condition": data["weather"][0]["description"]
            }
        return {"error": "City not found"}


#### Memory Agent


In [ ]:
class MemoryAgent:
    def __init__(self):
        self.memory = []

    def store(self, city, weather_info):
        self.memory.append({"city": city, "weather": weather_info})

    def recall(self):
        return self.memory


#### Response Agent


In [ ]:
class ResponseAgent:
    def generate_response(self, city, weather_info):
        if "error" in weather_info:
            return weather_info["error"]
        return (f"The current weather in {city} is {weather_info['condition']} with "
                f"a temperature of {weather_info['temperature']}°C and humidity of "
                f"{weather_info['humidity']}%.")


#### Full Workflow


In [ ]:
# Initialize agents
weather_agent = WeatherRetrievalAgent(WEATHER_API_KEY)
memory_agent = MemoryAgent()
response_agent = ResponseAgent()

# User input
city = input("Enter city name: ")
weather_info = weather_agent.get_weather(city)
memory_agent.store(city, weather_info)
response = response_agent.generate_response(city, weather_info)

print(response)
print("Memory:", memory_agent.recall())


## Agent Type 2: The Daily Dish Chatbot

Here's your challenge: You've been hired by "The Daily Dish," a popular restaurant with a customer service problem. Chef Maria, the owner, needs your help: "My team spends hours answering the same questions about reservations and menus," she explains. "Build me a chatbot that feels personal, to handle these inquiries so my staff can focus on creating exceptional dining experiences." Chef Maria sends you a PDF of frequently asked questions (FAQs) and now you must get the job done!


You'll solve this by creating specialized AI agents that work together to answer questions that users ask the chatbot! 


### **Project overview: Our chatbot's workflow**

Our chatbot will follow a four-step process to answer a user's question:

1. **Query Understanding Phase (Agent 1):**
The user’s question is first handled by a query understanding agent. This agent’s role is to interpret the user’s intent and extract key details from the question.

2. **Document Retrieval Phase (Agent 2):**
The interpreted query is used to search the restaurant’s FAQ PDF. This agent’s role is to retrieve only the most relevant sections of the document.

3. **Memory Phase (Agent 3):**
The retrieved information and user interaction are passed to a memory agent. This agent’s role is to store and recall conversation context, supporting more personalized responses.

4. **Response Generation Phase (Agent 4 – LLM):**
The retrieved FAQ content, along with memory context, is sent to an LLM-based response agent. This agent’s role is to generate a clear, friendly, and complete answer for the customer.

```
┌──────────────┐
│   User       │
│ (Question)   │
└──────┬───────┘
       │
       ▼
┌───────────────────────────┐
│ Query Understanding       │
│ Agent (Agent 1)           │
│ - Interprets user intent  │
│ - Extracts key keywords   │
└──────┬────────────────────┘
       │
       ▼
┌───────────────────────────┐
│ Document Retrieval        │
│ Agent (Agent 2)           │
│ - Searches FAQ PDF        │
│ - Retrieves relevant text │
└──────┬────────────────────┘
       │
       ▼
┌───────────────────────────┐
│ Memory Agent              │
│ (Agent 3)                 │
│ - Stores past questions   │
│ - Recalls conversation    │
│   context                 │
└──────┬────────────────────┘
       │
       ▼
┌───────────────────────────┐
│ LLM Response Agent        │
│ (Agent 4)                 │
│ - Combines retrieved info │
│ - Uses memory context     │
│ - Generates friendly reply│
└──────┬────────────────────┘
       │
       ▼
┌──────────────┐
│   User       │
│ (Final Reply)│
└──────────────┘
```


---


#### **Step 1 — Install Prerequisites**


In [ ]:
!pip install pypdf scikit-learn nltk

<span style="color:red"><b>Restart the kernel once the installation is complete.</b></span>


#### **Step 2 — Import Libraries**


In [ ]:
import re
import nltk
import numpy as np
from PyPDF2 import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('punkt')

# Ensure the FAQ PDF is in the same directory or provide the full path
# You can also download it from the source provided in the example notebook
# 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/7vgNfis17dQfjHAiIKkBOg/The-Daily-Dish-FAQ.pdf'
faq_pdf_path = "The_Daily_Dish_FAQ.pdf"

---


#### **Step 3 — Split the Document into Chunks**


### **Conclusion**

Congratulations! You've successfully built a basic RAG chatbot from scratch using Python and the OpenAI API. You've seen how to implement a **retrieval agent** to find relevant context and a **generative agent** to craft a final response. This hands-on experience provides a strong foundation for understanding more complex AI agent frameworks and building your own custom LLM-powered applications.
